CleanSIAPMonthlydata.py -- sayrejay@gmail.com

2023-11-16: When monthly data is yearly, the new_harv or new_planted variable is 0. This is probably what we want.

In [1]:
import pandas as pd
import os
import unicodedata
import numpy as np
from math import isnan
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import itertools
import matplotlib.pyplot as plt
import seaborn as sns

### Programs
def remove_accents(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    only_ascii = nfkd_form.encode('ASCII', 'ignore')
    return str(only_ascii)[:-1].replace("b'","")

def compute_fuzzy_match(input_df,input_col,key_df,key_col,key_code_col,output_col_name,output_col_code,threshold=86):
    from_matches = list(input_df[input_col].unique())
    to_matches = list(key_df[key_col].unique())
    codedict = dict(zip(list(key_df[key_col]),list(key_df[key_code_col])))

    matched_names, matched_accuracy, matched_code = [], [], []
    for mtch in from_matches[:]:
        if mtch not in to_matches:
            p = process.extract(mtch, to_matches,limit=1)
            if p[0][1] > threshold:            
                matched_names.append(p[0][0])
                matched_accuracy.append(p[0][1])
                matched_code.append(codedict[p[0][0]])
            else:
                matched_names.append("")
                matched_accuracy.append(0)
                matched_code.append("0")
        else:
            matched_names.append(mtch)
            matched_accuracy.append(100)
            matched_code.append(codedict[mtch])
    
    ### Build dicts to match each name with matched name, code, and accuracy score
    matched_names_dict = dict(zip(from_matches,matched_names))
    matched_accuracy_dict = dict(zip(from_matches,matched_accuracy))
    matched_code_dict = dict(zip(from_matches,matched_code))

    ### Write outputs back to df
    input_df.loc[:,output_col_name] = input_df[input_col].apply(lambda x: matched_names_dict.get(x))
    input_df.loc[:,"Fuzzywuzzy_match_accuracy"] = input_df[input_col].apply(lambda x: matched_accuracy_dict.get(x)) 
    input_df.loc[:,output_col_code] = input_df[input_col].apply(lambda x: matched_code_dict.get(x))
    return input_df
            
def fill_in(df, id_cols):
    """Fill in empty records for combinations of id_cols that do not exist
    in dataset.
    
    Args:
        df: dataset
        id_cols: list of identity columns

    Returns:
        filled_df: dataframe with empty records for missing combinations of id_cols
    """
    # create all possible unique combinations of id_cols
    # and find combos that do not exist in the dataset
    id_combos = list(itertools.product(*[df[c].unique() for c in id_cols]))
    existing_combos = df[id_cols].apply(tuple, axis=1).unique()
    missing_combos = set(id_combos) - set(existing_combos)

    # create an empty dataframe with the missing combos
    other_cols = [c for c in df.columns if c not in id_cols]
    new_idx = pd.MultiIndex.from_tuples(missing_combos, names=id_cols)
    empty_data = np.empty(shape=(len(missing_combos), len(other_cols))).fill(np.nan)
    filled_df = pd.DataFrame(data=empty_data, index=new_idx, columns=other_cols).reset_index()

    # concat dataset with empty dataset for missing combos
    return pd.concat([df.assign(_fill_in=0), filled_df.assign(_fill_in=1)]) 

def filter_above_6(input_lst):
    new_list = []
    for i in input_lst:
        if i > 6:
            new_list.append(i-12)
        else:
            new_list.append(i)
    return new_list

### Directories
base_dir         =  os.path.join(os.path.expanduser("~"), "Dropbox", "Projects", "Maize_prediction")
data_dir         =  os.path.join(base_dir, "Data",  "SIAP_monthly", "Input")
output_dir       =  os.path.join(base_dir, "Data",  "SIAP_monthly", "Output")
inegi_mun_dir    =  os.path.join(base_dir, "Data",  "muncodes/")
plot_dir         =  os.path.join(base_dir, "plots", "monthly_prod/")

### Inputs
muncodes         =  os.path.join(inegi_mun_dir,"muncodes_clean.xlsx")
localities       =  os.path.join(base_dir, "Data", "INEGI", "locality_info.csv")   # INEGI locality catalog (fuzzy-match fallback)

### Intermediates


### Outputs
monthly_siap_xls        =  os.path.join(output_dir,"mnthly_siap.xlsx")
monthly_siap_dta        =  os.path.join(output_dir,"mnthly_siap.dta")
max_harv_mth_xls        =  os.path.join(output_dir,"max_harv_mnth.xlsx")
max_harv_mth_dta        =  os.path.join(output_dir,"max_harv_mnth.dta")
max_harv_mth_ac_xls     =  os.path.join(output_dir,"max_harv_mnth_allcycles.xlsx")
max_harv_mth_ac_dta     =  os.path.join(output_dir,"max_harv_mnth_allcycles.dta")
max_harv_mth_agdist_xls =  os.path.join(output_dir,"max_harv_mnth_allcycles_agdist.xlsx")
max_harv_mth_agdist_dta =  os.path.join(output_dir,"max_harv_mnth_allcycles_agdist.dta")
max_harv_mth_by_yr_xls  =  os.path.join(output_dir,"max_harv_mnth_by_year.xlsx")
max_harv_mth_by_yr_dta  =  os.path.join(output_dir,"max_harv_mnth_by_year.dta")

/home/jdesktop/miniforge3/envs/geo_env/lib/python3.11/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
monthly_df = pd.DataFrame()
for r,d,f in os.walk(data_dir):
    if r == data_dir:
        for file in f:
            if '.csv' in file:
                try:
                    input_df = pd.read_csv(os.path.join(r, file))
                    monthly_df = pd.concat([monthly_df,input_df])
                except:
                    print(os.path.join(r, file))
                    
monthly_df.loc[:,'estado']    =  monthly_df['Ent'].apply(lambda x: remove_accents(x).capitalize())
monthly_df.loc[:,'municipio'] =  monthly_df['Mun'].apply(lambda x: remove_accents(x).capitalize())
monthly_df.drop(['Ent','Mun'],axis=1,inplace=True)
monthly_df.loc[:,'estado']    =  monthly_df['estado'].apply(lambda x: 'Coahuila de zaragoza' if x == 'Coahuila' else x)
monthly_df.loc[:,'estado']    =  monthly_df['estado'].apply(lambda x: 'Michoacan de ocampo' if x == 'Michoacan' else x)
monthly_df.loc[:,'estado']    =  monthly_df['estado'].apply(lambda x: 'Veracruz de ignacio de la llave' if x == 'Veracruz' else x)
name_df                       =  monthly_df[['estado','municipio']].drop_duplicates().reset_index(drop=True)

In [3]:
localities_df = pd.read_csv(localities,dtype='str')
localities_df.loc[:,'nom_ent']  = localities_df['nom_ent'].apply(lambda x: remove_accents(x).capitalize())
localities_df.loc[:,'nom_ent']  = localities_df['nom_ent'].apply(lambda x: 'Ciudad de mexico' if x == 'Distrito federal' else x)
localities_df.loc[:,'nom_ent']  = localities_df['nom_ent'].apply(lambda x: 'Michoacan de ocampo' if x == 'Michoacan' else x)
inegi_codes_df = pd.read_excel(muncodes,dtype='str')

In [4]:
states = list(inegi_codes_df['Estado'].unique())
df     = pd.DataFrame()
for st in states[:]:
    name_st_df  = name_df[name_df['estado'] == st].copy()
    inegi_st_df = inegi_codes_df[inegi_codes_df['Estado'] == st].copy()
    name_st_df  = compute_fuzzy_match(name_st_df,'municipio',inegi_st_df,'Municipio','muncode','matched_mun','matched_muncode')
    df          = pd.concat([df,name_st_df])

unmatched_df    = df[df['Fuzzywuzzy_match_accuracy'] == 0].copy()
matched_df      = df[df['Fuzzywuzzy_match_accuracy'] != 0].copy()
unmatched_df.drop(['matched_mun','Fuzzywuzzy_match_accuracy','matched_muncode'],axis=1,inplace=True)

locality_matcheddf = pd.DataFrame()
for st in states[:]:
    unmatched_st_df    = unmatched_df[unmatched_df['estado'] == st].copy()
    localities_st_df   = localities_df[localities_df['nom_ent'] == st].copy()
    unmatched_st_df    = compute_fuzzy_match(unmatched_st_df,'municipio',localities_st_df,'nom_loc','CVE_GEO','matched_loc','matched_geocode')
    locality_matcheddf = pd.concat([locality_matcheddf,unmatched_st_df])
    
unmatched_df           = locality_matcheddf[locality_matcheddf['matched_geocode'] == '0'].copy()
locality_matcheddf     = locality_matcheddf[locality_matcheddf['matched_geocode'] != '0'].copy()
locality_matcheddf['matched_muncode'] = locality_matcheddf['matched_geocode'].apply(lambda x: x[:5])

unmatched_df['matched_geocode'] = unmatched_df['matched_geocode'].apply(lambda x: "20549" if x=='0'  else x)
unmatched_df['matched_loc']     = unmatched_df['matched_geocode'].apply(lambda x: "Tezoatlan de segura y luna" if x=='20549'  else x)
unmatched_df['Fuzzywuzzy_match_accuracy'] = 100
unmatched_df.columns            = ['estado','municipio','matched_mun','Fuzzywuzzy_match_accuracy','matched_muncode']

mundict = dict(zip(list(inegi_codes_df['muncode']),list(inegi_codes_df['Municipio'])))
locality_matcheddf['matched_mun'] = locality_matcheddf['matched_muncode'].apply(lambda x: mundict.get(x))

df = pd.concat([matched_df,unmatched_df])
df = pd.concat([df,locality_matcheddf[['estado','municipio','matched_mun','Fuzzywuzzy_match_accuracy','matched_muncode']]])
# ### Add matched states
st_dict           = dict(zip(list(inegi_codes_df['muncode']),list(inegi_codes_df['Estado'])))
df['matched_est'] = df['matched_muncode'].apply(lambda x: st_dict.get(x))
df                = df.reset_index(drop=True)

In [5]:
### Merge monthly data with matched muncodes
mnth_df    = monthly_df.merge(df,on=['estado','municipio'], how='left')
mnth_df.drop(['estado','municipio','Fuzzywuzzy_match_accuracy'],axis=1,inplace=True)
reorg_cols = ['matched_est','matched_mun','Crop','Mes','Year','Irrig','Cycle','Sup. Sem.','Sup. Cos.','Sup. Sin.','Prod','Rendi','Just_planted','matched_muncode']
mnth_df    = mnth_df[reorg_cols]

mnth_df.rename(columns={'matched_mun':'Municipio','matched_muncode':'muncode','Year':'year',
                                       'matched_est':'Estado','Sup. Sem.':'ha_planted','Sup. Cos.':'ha_harv',
                                      'Sup. Sin.':'ha_lost','Prod':'q','Rendi':'yield'}, inplace=True)
mnth_df    = mnth_df.sort_values(['Crop','muncode','Irrig','Cycle','year','Mes']).reset_index(drop=True)

### Change irrig status to binary
mnth_df['Irrig']      = mnth_df['Irrig'].apply(lambda x: 1 if x == 'Riego' else 0)

mnth_name_df          = mnth_df[['Estado','Municipio','muncode']].drop_duplicates()
mnth_df               = mnth_df.drop(['Estado','Municipio'],axis=1)

crop_to_num_dict      = dict([(crop, str(i+1).zfill(3)) for i, crop in enumerate(mnth_df['Crop'].unique())])
num_to_crop_dict      = dict([(str(i+1).zfill(3), crop) for i, crop in enumerate(mnth_df['Crop'].unique())])

### Convert crop cycle to code
cycle_to_num_dict     = {'Perennes':3, 'Otoño - Invierno':1, 'Primavera - Verano':2}
mnth_df['Cycle']      = mnth_df['Cycle'].replace(cycle_to_num_dict)
mnth_df['cropcode']   = mnth_df['Crop'].apply(lambda x: crop_to_num_dict[x])
mnth_df['uniqcode']   = 'M'+mnth_df['muncode']+'C'+mnth_df['cropcode'].astype(str)+'I'+mnth_df['Irrig'].astype(str)+'S'+mnth_df['Cycle'].astype(str)
mnth_df['Mes']        = mnth_df['Mes'].apply(lambda x: '0'+str(int(x)) if len(str(int(x))) == 1 else str(int(x)))
mnth_df['timestr']    = mnth_df['year'].astype(int).astype(str)+mnth_df['Mes']
mnth_df               = fill_in(mnth_df, ['uniqcode', 'timestr'])
mnth_df               = mnth_df.sort_values(['uniqcode','timestr'])

mnth_df['muncode']    = mnth_df['muncode'].fillna(mnth_df['uniqcode'].apply(lambda x: x[1:6]))
mnth_df['cropcode']   = mnth_df['cropcode'].fillna(mnth_df['uniqcode'].apply(lambda x: x[7:10]))
mnth_df['year']       = mnth_df['year'].fillna(mnth_df['timestr'].apply(lambda x: x[:4]))
mnth_df['Mes']        = mnth_df['Mes'].fillna(mnth_df['timestr'].apply(lambda x: x[4:]))
mnth_df['Crop']       = mnth_df['Crop'].fillna(mnth_df['cropcode'].apply(lambda x: num_to_crop_dict[x]))
mnth_df['Irrig']      = mnth_df['Irrig'].fillna(mnth_df['uniqcode'].apply(lambda x: x[11:12]))
mnth_df['Cycle']      = mnth_df['Cycle'].fillna(mnth_df['uniqcode'].apply(lambda x: x[13:14]))
mnth_df               = mnth_df.reset_index(drop=True)

for i in list(mnth_df.index)[:-1]:
    if mnth_df.loc[i,'uniqcode'] == mnth_df.loc[i+1,'uniqcode']:
        ### Create one zero just before just planted comes in 
        if str(mnth_df.loc[i,'Just_planted'])=='nan':
            if str(mnth_df.loc[i+1,'Just_planted'])!='nan' and mnth_df.loc[i+1,'Just_planted'] != 0:
                mnth_df.loc[i,'Just_planted'] = 0.0
        ### Fill forward q, ha_harv
        for var in ['ha_harv','q']:
            if str(mnth_df.loc[i,var]) != 'nan' and str(mnth_df.loc[i+1,var]) == 'nan':
                mnth_df.loc[i+1,var] = mnth_df.loc[i,var]

In [6]:
mnth_df['ha_planted']   = mnth_df['Just_planted'].fillna(mnth_df['ha_planted'])

mnth_df['new_harv']     = mnth_df['ha_harv']-mnth_df['ha_harv'].shift(1)
mnth_df['new_q']        = mnth_df['q']-mnth_df['q'].shift(1)
mnth_df['new_plants']   = mnth_df['ha_planted']-mnth_df['ha_planted'].shift(1)

# ### Fill in obs taken erroneously from different crop-muns
mnth_df.loc[list(mnth_df[mnth_df['timestr'] == '201801'].index),'new_harv'] = np.nan
mnth_df.loc[list(mnth_df[mnth_df['timestr'] == '201801'].index),'new_q'] = np.nan
mnth_df.loc[list(mnth_df[mnth_df['timestr'] == '201801'].index),'new_plants'] = np.nan

# ### Fill in negatives
mnth_df['new_harv']     = mnth_df['new_harv'].apply(lambda x: np.nan if x < 0 else x)
mnth_df['new_q']        = mnth_df['new_q'].apply(lambda x: np.nan if x < 0 else x)
mnth_df['new_plants']   = mnth_df['new_plants'].apply(lambda x: np.nan if x < 0 else x)
### Fill in negatives with new amount
mnth_df['new_harv']     = mnth_df['new_harv'].fillna(mnth_df['ha_harv'])
mnth_df['new_q']        = mnth_df['new_q'].fillna(mnth_df['q'])
mnth_df['new_plants']   = mnth_df['new_plants'].fillna(mnth_df['ha_planted'])

### 2023-11-16: Check for whether production that is supposedly monthly is actually annual. 
### I don't think that anything should be done in these cases, because ha_harv == 0 and ha_planted == 0 in these cases.
### So the alternative is that I make some assumptions about the distribution of production across months, which I don't want to do.
###grouped_df                 = mnth_df.groupby("uniqcode")
#### Check if "ha_harv" is the same across a twelve month period
###ha_harv_same               = grouped_df["ha_harv"].rolling(window=12).apply(lambda x: len(set(x)) == 1, raw=False)
###ha_planted_same            = grouped_df["ha_planted"].rolling(window=12).apply(lambda x: len(set(x)) == 1, raw=False)
#### Add the result as a new column to the dataframe
###mnth_df["ha_harv_same"]    = ha_harv_same.reset_index(drop=True)
###mnth_df["ha_plant_same"]   = ha_planted_same.reset_index(drop=True)
###mnth_df[mnth_df["ha_harv_same"] == 1]['ha_harv'].sum()
###mnth_df[mnth_df["ha_plant_same"] == 1]['ha_planted'].sum()

mnth_df                 = mnth_df.merge(mnth_name_df, on = 'muncode', how = 'left')
mnth_df                 = mnth_df.drop(['cropcode','uniqcode','timestr'], axis=1)
mnth_df['year']         = mnth_df['year'].astype(int)

mnth_df                 = mnth_df[['Crop','Irrig','Cycle','year','Mes','muncode','Estado','Municipio','new_harv','new_q','new_plants']]
mnth_df.columns         = ['Crop','Irrig','Cycle','year','Mes','muncode','Estado','Municipio','ha_harv','q','ha_planted']
### 2026-08-27: cast to float before dividing — object-dtype division raises
### ZeroDivisionError under pandas 3 (floats give inf/nan, handled below)
mnth_df['q']            = mnth_df['q'].astype(float)
mnth_df['ha_harv']      = mnth_df['ha_harv'].astype(float)
mnth_df['yield']        = mnth_df['q']/mnth_df['ha_harv']
mnth_df['yield']        = mnth_df['yield'].apply(lambda x: np.nan if np.isinf(x) else x)

mnth_df['q']            = mnth_df['q'].astype(float)
mnth_df['yield']        = mnth_df['yield'].astype(float)
mnth_df['ha_planted']   = mnth_df['ha_planted'].astype(float)
mnth_df['ha_harv']      = mnth_df['ha_harv'].astype(float)
mnth_df['Mes']          = mnth_df['Mes'].astype(int)
mnth_df['Irrig']        = mnth_df['Irrig'].astype(int)
mnth_df['Cycle']        = mnth_df['Cycle'].astype(int)

for prod_col in ['ha_harv','q','ha_planted','yield']:
    mnth_df[prod_col]   = mnth_df[prod_col].apply(lambda x: np.nan if x == 0 else x)

mnth_df                 = mnth_df.dropna(subset=['ha_harv','q','ha_planted','yield'], how='all')

### Remove January 2018, which is the first month in the dataset and thus meaningless
mnth_df                = mnth_df[~((mnth_df['year']==2018)&(mnth_df['Mes']==1))]

### Write to output
mnth_df.to_stata(monthly_siap_dta, write_index=False)
mnth_df.to_excel(monthly_siap_xls, index=False)

### Compute max month for harvest, planting, and production

In [7]:
type_agg = "separate" ### Compute max month for harvest, planting, and production separately for irrigated and non-irrigated land
# type_agg = "all_land" ### All land together (irrig+non irrig)
# type_agg = "ag_dist"  ### all land together (irrig+non irrig) by agricultural district

In [8]:
mnth_df                =  pd.read_stata(monthly_siap_dta)
### Remove January 2018, which is the first month in the dataset and thus meaningless
mnth_df                = mnth_df[~((mnth_df['year']==2018)&(mnth_df['Mes']==1))]
### 2024 doesn't have data for all months, so remove for purposes of calculating max months
mnth_df                =  mnth_df[mnth_df['year'] != 2024]

if type_agg == "separate":
    mnth_group_vars        =  ['muncode', 'Crop', 'Estado', 'Municipio','Irrig','Cycle']
elif type_agg == "all_land":
    mnth_group_vars        =  ['muncode', 'Crop', 'Estado', 'Municipio','Cycle']
    mnth_df = mnth_df.groupby(['year','Mes']+mnth_group_vars)[['ha_harv','q','ha_planted']].sum().reset_index()
elif type_agg == "ag_dist":
    mnth_group_vars        =  ['ag_dist', 'Crop', 'Cycle']
    st_to_agdict = {'Aguascalientes': 'Centro Occidente', 'Baja california': 'Noroeste', 'Baja california sur': 'Noroeste',
        'Campeche': 'Sur-Sureste', 'Coahuila de zaragoza': 'Noreste', 'Colima': 'Centro Occidente', 'Chiapas': 'Sur-Sureste', 'Chihuahua': 'Noreste',
        'Ciudad de mexico': 'Centro', 'Durango': 'Noreste', 'Guanajuato': 'Centro Occidente', 'Guerrero': 'Centro',
        'Hidalgo': 'Centro', 'Jalisco': 'Centro Occidente', 'Mexico': 'Centro', 'Michoacan de ocampo': 'Centro Occidente',
        'Morelos': 'Centro', 'Nayarit': 'Noroeste', 'Nuevo leon': 'Noreste', 'Oaxaca': 'Sur-Sureste',
        'Puebla': 'Centro', 'Queretaro': 'Centro Occidente', 'Quintana roo': 'Sur-Sureste', 'San luis potosi': 'Centro Occidente',
        'Sinaloa': 'Noroeste', 'Sonora': 'Noroeste', 'Tabasco': 'Sur-Sureste', 'Tamaulipas': 'Noreste',
        'Tlaxcala': 'Centro', 'Veracruz de ignacio de la llave': 'Sur-Sureste', 'Yucatan': 'Sur-Sureste', 'Zacatecas': 'Noreste'}
    mnth_df['ag_dist'] = mnth_df['Estado'].apply(lambda x: st_to_agdict[x])
    mnth_df = mnth_df.groupby(['year','Mes']+mnth_group_vars)[['ha_harv','q','ha_planted']].sum().reset_index()

# Find the index of maximum for each group
idx_max_harv           =  mnth_df.dropna(subset=['ha_harv']).groupby(['year']+mnth_group_vars)['ha_harv'].idxmax()
idx_max_plants         =  mnth_df.dropna(subset=['ha_planted']).groupby(['year']+mnth_group_vars)['ha_planted'].idxmax()
idx_max_q              =  mnth_df.dropna(subset=['q']).groupby(['year']+mnth_group_vars)['q'].idxmax()

# Filter NaN from the indices
idx_max_harv           =  idx_max_harv.dropna().astype(int)
idx_max_plants         =  idx_max_plants.dropna().astype(int)
idx_max_q              =  idx_max_q.dropna().astype(int)

# Extract the 'Mes' values for these indices
max_harv_mes           =  mnth_df.loc[idx_max_harv,   ['year']+mnth_group_vars+['Mes']]
max_plants_mes         =  mnth_df.loc[idx_max_plants, ['year']+mnth_group_vars+['Mes']]
max_q_mes              =  mnth_df.loc[idx_max_q,      ['year']+mnth_group_vars+['Mes']]

max_harv_mes.columns   =  ['year']+mnth_group_vars+['max_harv_month']
max_plants_mes.columns =  ['year']+mnth_group_vars+['max_plants_month']
max_q_mes.columns      =  ['year']+mnth_group_vars+['max_q_month']


max_month_df           =  max_harv_mes.merge(max_plants_mes, on = ['year']+mnth_group_vars, how = 'outer')
max_month_df           =  max_month_df.merge(max_q_mes,      on = ['year']+mnth_group_vars, how = 'outer')

### Write this intermediate output to file
if type_agg == "separate":
    max_month_df.to_excel(max_harv_mth_by_yr_xls, index=False)
    max_month_df.to_stata(max_harv_mth_by_yr_dta, write_index=False)

max_harv_list_df       =  max_month_df.groupby(mnth_group_vars)['max_harv_month'].apply(list).reset_index()
max_plant_list_df      =  max_month_df.groupby(mnth_group_vars)['max_plants_month'].apply(list).reset_index()
max_q_list_df          =  max_month_df.groupby(mnth_group_vars)['max_q_month'].apply(list).reset_index()

max_harv_list_df['max_harv_month']     =  max_harv_list_df['max_harv_month'].apply(lambda x: [int(i) for i in x if pd.notnull(i)])
max_plant_list_df['max_plants_month']  =  max_plant_list_df['max_plants_month'].apply(lambda x: [int(i) for i in x if pd.notnull(i)])
max_q_list_df['max_q_month']           =  max_q_list_df['max_q_month'].apply(lambda x: [int(i) for i in x if pd.notnull(i)])

max_harv_list_df       =  max_harv_list_df[max_harv_list_df['max_harv_month'].apply(lambda x: len(x) > 0)]
max_plant_list_df      =  max_plant_list_df[max_plant_list_df['max_plants_month'].apply(lambda x: len(x) > 0)]
max_q_list_df          =  max_q_list_df[max_q_list_df['max_q_month'].apply(lambda x: len(x) > 0)]

max_list_df            = max_harv_list_df.merge(max_plant_list_df, on = mnth_group_vars, how = 'outer')
max_list_df            = max_list_df.merge(max_q_list_df,          on = mnth_group_vars, how = 'outer')

for max_mth_var in ['max_harv_month','max_plants_month','max_q_month']:
    max_var_nonan_i = list(max_list_df[max_list_df[max_mth_var].apply(lambda x: str(x) != 'nan')].index)
    max_list_df.loc[max_var_nonan_i,max_mth_var]           =  max_list_df.loc[max_var_nonan_i,max_mth_var].apply(lambda x: filter_above_6(x) if (min(x) <= 3 and max(x) >= 10)  else x)
    max_list_df.loc[max_var_nonan_i,max_mth_var+'_median'] =  max_list_df.loc[max_var_nonan_i,max_mth_var].apply(lambda x: np.median(x))
    max_list_df.loc[max_var_nonan_i,max_mth_var+'_median'] =  max_list_df.loc[max_var_nonan_i,max_mth_var+'_median'].apply(lambda x: np.floor(x)-1)
    max_list_df.loc[max_var_nonan_i,max_mth_var+'_median'] =  max_list_df.loc[max_var_nonan_i,max_mth_var+'_median'].apply(lambda x: x+12 if x < 1 else x)

max_list_df = max_list_df[mnth_group_vars+['max_harv_month_median', 'max_plants_month_median','max_q_month_median']]
if type_agg == "separate":
    max_list_df.to_excel(max_harv_mth_xls, index=False)
    max_list_df.to_stata(max_harv_mth_dta, write_index=False)
elif type_agg == "all_land":
    max_list_df.to_excel(max_harv_mth_ac_xls, index=False)
    max_list_df.to_stata(max_harv_mth_ac_dta, write_index=False)
elif type_agg == "ag_dist":
    max_list_df.to_excel(max_harv_mth_agdist_xls, index=False)
    max_list_df.to_stata(max_harv_mth_agdist_dta, write_index=False)

Plot the seasonality of maize planting

In [9]:

# ## Plot harvesting by month in all of Mexico
# maiz_plant_df = mnth_df[mnth_df['Crop'] == 'Maíz grano']
# maiz_plant_df = maiz_plant_df.groupby(['Mes','muncode','Cycle'])[['ha_planted','ha_harv']].mean().reset_index()
# maiz_plant_df = maiz_plant_df.groupby(['Mes','Cycle'])[['ha_planted','ha_harv']].sum().reset_index()
# maiz_plant_df['ha_planted'] = maiz_plant_df['ha_planted']/1000000
# maiz_plant_df['ha_harv']    = maiz_plant_df['ha_harv']/1000000

# for crop_cycle in maiz_plant_df['Cycle'].unique():
#     maiz_plant_cyc_df = maiz_plant_df[maiz_plant_df['Cycle'] == crop_cycle]
#     if crop_cycle == 3:
#         crop_cycle, cycle_fl_name = 'Perennial', 'per'
#     elif crop_cycle == 1:
#         crop_cycle, cycle_fl_name = 'Fall-Winter', 'fw'
#     elif crop_cycle == 2:
#         crop_cycle, cycle_fl_name = 'Spring-Summer', 'sp_su'
#     ### Plotting code
#     fig, ax = plt.subplots(1,1, figsize=(14, 7))

#     rects = ax.bar(range(12),maiz_plant_cyc_df['ha_planted'], width=0.25, label='Hectares planted', color='#70ad47')
#     rects2 = ax.bar([y+0.25 for y in range(12)],maiz_plant_cyc_df['ha_harv'], width=0.25, label='Hectares harvested', color='#ffc000')
#     # # ['#00b0f0','fuchsia','mediumpurple','#ed7d31','#70ad47','#ffc000','red']

#     # ax.set_yticks([0,0.25,0.5,0.75,1,1.25,1.5,1.75,2,2.25])
#     ax.set_title('Average maize planting and harvesting by month in Mexico for '+crop_cycle+' maize, 2018-2022')
#     ax.set_xticks(range(12),["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
#     ax.set_ylabel("Millions of hectares")#, fontsize = 12, rotation = 0)
#     ax.set_xlabel("Month")
#     plt.legend(bbox_to_anchor =(0.5,-0.15), loc='lower center', frameon=False, ncol=7)
#     # ax.yaxis.set_label_coords(0.04, 1.02)
#     plt.xticks(rotation = 0)
#     ax.spines.right.set_visible(False)
#     ax.spines.top.set_visible(False)
#     plt.savefig(os.path.join(plot_dir,'maize_monthly_harvesting_'+cycle_fl_name+'.png'),bbox_inches='tight',dpi=600)
#     plt.show()

# ### Plot harvesting by month by state
# maiz_est_plant_df               = mnth_df[mnth_df['Crop'] == 'Maíz grano']
# maiz_est_plant_df               = maiz_est_plant_df.groupby(['Mes','muncode','Estado','Cycle'])[['ha_planted','ha_harv']].mean().reset_index()
# maiz_est_plant_df               = maiz_est_plant_df.groupby(['Mes','Estado','Cycle'])[['ha_planted','ha_harv']].sum().reset_index()
# maiz_est_plant_df['ha_planted'] = maiz_est_plant_df['ha_planted']/1000
# maiz_est_plant_df['ha_harv']   = maiz_est_plant_df['ha_harv']/1000


# for crop_cycle in maiz_est_plant_df['Cycle'].unique():
#     for est in maiz_est_plant_df['Estado'].unique():
        
#         maiz_this_est_plant_df = maiz_est_plant_df[(maiz_est_plant_df['Estado'] == est) & (maiz_est_plant_df['Cycle'] == crop_cycle)]
#         if len(maiz_this_est_plant_df) < 12 and len(maiz_this_est_plant_df) != 0:
#             all_months_df = pd.DataFrame({'Mes': range(1, 13)})
#             all_months_df['Estado'] = est
#             all_months_df['Cycle']  = crop_cycle
#             maiz_this_est_plant_df = pd.merge(all_months_df, maiz_this_est_plant_df, on=['Mes','Estado','Cycle'], how='outer').fillna(0)
#             maiz_this_est_plant_df = maiz_this_est_plant_df.sort_values('Mes').reset_index(drop=True)

#         if crop_cycle == 3:
#             crop_cycle_nm, cycle_fl_name = 'Perennial', 'per'
#         elif crop_cycle == 1:
#             crop_cycle_nm, cycle_fl_name = 'Fall-Winter', 'fw'
#         elif crop_cycle == 2:
#             crop_cycle_nm, cycle_fl_name = 'Spring-Summer', 'sp_su'

#         if len(maiz_this_est_plant_df) > 0:
#             fig, ax = plt.subplots(1,1, figsize=(14, 7))

#             rects  = ax.bar(range(12),                  maiz_this_est_plant_df['ha_planted'], width=0.25, label='Hectares planted',   color='#70ad47')
#             rects2 = ax.bar([y+0.25 for y in range(12)],maiz_this_est_plant_df['ha_harv'],    width=0.25, label='Hectares harvested', color='#ffc000')

#             # ax.set_yticks([0,0.25,0.5,0.75,1,1.25,1.5,1.75,2,2.25])
#             ax.set_title('Average maize planting and harvesting by month in '+est+' for '+crop_cycle_nm+' season, 2018-2022')
#             ax.set_xticks(range(12),["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
#             ax.set_ylabel("Thousands of hectares")#, fontsize = 12, rotation = 0)
#             ax.set_xlabel("Month")
#             plt.legend(bbox_to_anchor =(0.5,-0.15), loc='lower center', frameon=False, ncol=7)
#             #ax.yaxis.set_label_coords(0.04, 1.02)
#             plt.xticks(rotation = 0)
#             ax.spines.right.set_visible(False)
#             ax.spines.top.set_visible(False)
#             plt.savefig(os.path.join(plot_dir,'maize_monthly_harvesting'+remove_accents(est).replace(' ','_')+'_'+cycle_fl_name+'.png'),bbox_inches='tight',dpi=600)

# for yr in range(2018,2023):
#     ### Plot harvesting by month by state
#     maiz_est_plant_df               = mnth_df[mnth_df['Crop'] == 'Maíz grano']
#     maiz_est_plant_df               = maiz_est_plant_df[maiz_est_plant_df['year'] == yr]
#     maiz_est_plant_df               = maiz_est_plant_df.groupby(['Mes','Estado','Cycle'])[['ha_planted','ha_harv']].sum().reset_index()
#     maiz_est_plant_df['ha_planted'] = maiz_est_plant_df['ha_planted']/1000
#     maiz_est_plant_df['ha_harv']   = maiz_est_plant_df['ha_harv']/1000

#     for crop_cycle in maiz_est_plant_df['Cycle'].unique():
#         for est in maiz_est_plant_df['Estado'].unique():
#             maiz_this_est_plant_df = maiz_est_plant_df[(maiz_est_plant_df['Estado'] == est) & (maiz_est_plant_df['Cycle'] == crop_cycle)]
#             if len(maiz_this_est_plant_df) < 12 and len(maiz_this_est_plant_df) != 0:
#                 all_months_df = pd.DataFrame({'Mes': range(1, 13)})
#                 all_months_df['Estado'] = est
#                 all_months_df['Cycle']  = crop_cycle
#                 maiz_this_est_plant_df = pd.merge(all_months_df, maiz_this_est_plant_df, on=['Mes','Estado','Cycle'], how='outer').fillna(0)
#                 maiz_this_est_plant_df = maiz_this_est_plant_df.sort_values('Mes').reset_index(drop=True)

#             if crop_cycle == 3:
#                 crop_cycle_nm, cycle_fl_name = 'Perennial', 'per'
#             elif crop_cycle == 1:
#                 crop_cycle_nm, cycle_fl_name = 'Fall-Winter', 'fw'
#             elif crop_cycle == 2:
#                 crop_cycle_nm, cycle_fl_name = 'Spring-Summer', 'sp_su'


#             if len(maiz_this_est_plant_df) > 0:
#                 fig, ax = plt.subplots(1,1, figsize=(14, 7))

#                 rects  = ax.bar(range(12),maiz_this_est_plant_df['ha_planted'], width=0.25, label='Hectares planted', color='#70ad47')
#                 rects2 = ax.bar([y+0.25 for y in range(12)],maiz_this_est_plant_df['ha_harv'], width=0.25, label='Hectares harvested', color='#ffc000')

#                 # ax.set_yticks([0,0.25,0.5,0.75,1,1.25,1.5,1.75,2,2.25])
#                 ax.set_title('Average maize planting and harvesting by month in '+est+' for '+crop_cycle_nm+' season, '+str(yr))
#                 ax.set_xticks(range(12),["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
#                 ax.set_ylabel("Thousands of hectares")#, fontsize = 12, rotation = 0)
#                 ax.set_xlabel("Month")
#                 plt.legend(bbox_to_anchor =(0.5,-0.15), loc='lower center', frameon=False, ncol=7)
#                 #ax.yaxis.set_label_coords(0.04, 1.02)
#                 plt.xticks(rotation = 0)
#                 ax.spines.right.set_visible(False)
#                 ax.spines.top.set_visible(False)
#                 plt.savefig(os.path.join(plot_dir,str(yr),'maize_monthly_harvesting'+remove_accents(est).replace(' ','_')+'_'+str(yr)+'_'+cycle_fl_name+'.png'),bbox_inches='tight',dpi=600)


### Plot seasonality of avocado harvest

In [10]:
mnth_df = pd.read_stata(monthly_siap_dta)

# for yr in range(2018,2024):
for yr in [2023, 2024]:
    ### Plot harvesting by month by state
    avo_est_plant_df               = mnth_df[mnth_df['Crop'] == 'Aguacate']
    avo_est_plant_df               = avo_est_plant_df[avo_est_plant_df['year'] == yr]
    avo_est_plant_df               = avo_est_plant_df.groupby(['Mes','Estado'])[['ha_planted','ha_harv']].sum().reset_index()
    avo_est_plant_df['ha_planted'] = avo_est_plant_df['ha_planted']/1000
    avo_est_plant_df['ha_harv']    = avo_est_plant_df['ha_harv']/1000

    all_months_plot_df = pd.DataFrame({'Mes': range(1, 13)})
    all_months_plot_df = pd.merge(all_months_plot_df, avo_est_plant_df.groupby('Mes').sum().reset_index(), on='Mes', how='outer').fillna(0)
    all_months_plot_df = all_months_plot_df.sort_values('Mes').reset_index(drop=True)


    fig, ax = plt.subplots(1,1, figsize=(14, 7))

    rects = ax.bar(range(12),all_months_plot_df['ha_planted'], width=0.25, label='Hectares planted', color='#70ad47')
    rects2 = ax.bar([y+0.25 for y in range(12)],all_months_plot_df['ha_harv'], width=0.25, label='Hectares harvested', color='#ffc000')

    # ax.set_yticks([0,0.25,0.5,0.75,1,1.25,1.5,1.75,2,2.25])
    ax.set_title('Average avocado planting and harvesting by month in for '+str(yr))
    ax.set_xticks(range(12),["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
    ax.set_ylabel("Thousands of hectares")#, fontsize = 12, rotation = 0)
    ax.set_xlabel("Month")
    plt.legend(bbox_to_anchor =(0.5,-0.15), loc='lower center', frameon=False, ncol=7)
    #ax.yaxis.set_label_coords(0.04, 1.02)
    plt.xticks(rotation = 0)
    ax.spines.right.set_visible(False)
    ax.spines.top.set_visible(False)
    plt.savefig(os.path.join(plot_dir,str(yr),'avocado_monthly_harvesting_'+str(yr)+'.png'),bbox_inches='tight',dpi=600)


    for est in avo_est_plant_df['Estado'].unique():
        avo_this_est_plant_df = avo_est_plant_df[avo_est_plant_df['Estado'] == est]
        if len(avo_this_est_plant_df) < 12 and len(avo_this_est_plant_df) != 0:
            all_months_df = pd.DataFrame({'Mes': range(1, 13)})
            all_months_df['Estado'] = est
            avo_this_est_plant_df = pd.merge(all_months_df, avo_this_est_plant_df, on=['Mes','Estado'], how='outer').fillna(0)
            avo_this_est_plant_df = avo_this_est_plant_df.sort_values('Mes').reset_index(drop=True)


        if len(avo_this_est_plant_df) > 0:
            fig, ax = plt.subplots(1,1, figsize=(14, 7))

            rects = ax.bar(range(12),avo_this_est_plant_df['ha_planted'], width=0.25, label='Hectares planted', color='#70ad47')
            rects2 = ax.bar([y+0.25 for y in range(12)],avo_this_est_plant_df['ha_harv'], width=0.25, label='Hectares harvested', color='#ffc000')

            # ax.set_yticks([0,0.25,0.5,0.75,1,1.25,1.5,1.75,2,2.25])
            ax.set_title('Average avocado planting and harvesting by month in '+est+' for '+str(yr))
            ax.set_xticks(range(12),["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
            ax.set_ylabel("Thousands of hectares")#, fontsize = 12, rotation = 0)
            ax.set_xlabel("Month")
            plt.legend(bbox_to_anchor =(0.5,-0.15), loc='lower center', frameon=False, ncol=7)
            #ax.yaxis.set_label_coords(0.04, 1.02)
            plt.xticks(rotation = 0)
            ax.spines.right.set_visible(False)
            ax.spines.top.set_visible(False)
            plt.savefig(os.path.join(plot_dir,str(yr),'avocado_monthly_harvesting'+remove_accents(est).replace(' ','_')+'_'+str(yr)+'.png'),bbox_inches='tight',dpi=600)
            plt.close('all')
